## Implementação de tradutor Inglês -> Alemão



In [1]:
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 115.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 114.0 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import random


# basicamente readile and readlines de um local do drive




# get a dataset from kaggle with translation
# english to german
# the strings are separated by a tab (\t)
# the file format is txt
# the lines are the txt file lines

In [ ]:
PAIRS_LEN = 200000

pairs = {'en': [], 'de': []}

for line in lines[:PAIRS_LEN]:
    pair = line.split('\t')
    pairs['en'].append(pair[0])
    pairs['de'].append(pair[1])

In [ ]:
len(pairs['en']), len(pairs['de'])

In [ ]:
TEST_PAIRS_LEN = 70000

test_pairs = {'en': [], 'de': []}

for line in lines[PAIRS_LEN:PAIRS_LEN + TEST_PAIRS_LEN]:
    pair = line.split('\t')
    test_pairs['en'].append(pair[0])
    test_pairs['de'].append(pair[1])

In [ ]:
test_pairs['en'][:5], test_pairs['de'][:5]

In [9]:
import spacy

nlp_de = spacy.load('de_core_news_sm')
nlp_en = spacy.load('en_core_web_sm')

doc = nlp_de(pairs['de'][0])
print([(token.text, token.lemma_, token.pos_) for token in doc])

NameError: name 'pairs' is not defined

In [ ]:
doc = nlp_de('I don\'t know what to do')
print([token for token in doc])

In [ ]:
doc = nlp_en('I don\'t know what to do')
print([token for token in doc])

In [ ]:
class Test():
    def __init__(self, x):
        self.x = x

# ???

In [ ]:
def tokenizer_de(text):
    return [str(token) for token in nlp_de.tokenizer(text)]

def tokenizer_en(text):
    return [str(token) for token in nlp_en.tokenizer(text)]

In [10]:
tokenizer_en('I don\'t know what to do')

['I', 'do', "n't", 'know', 'what', 'to', 'do']

In [11]:
def tokenizer_all(pairs):
    tokens = {'en': [], 'de': []}

    for pair in pairs['en']:
        tokens['en'].append(tokenizer_en(pair))

    for pair in pairs['de']:
        tokens['de'].append(tokenizer_de(pair))
    
    return tokens

In [ ]:
tokens = tokenizer_all(pairs) # 200k
test_tokens = tokenizer_all(test_pairs) # 70k

NameError: name 'pairs' is not defined

In [ ]:
pairs['en'][:5], tokens['en'][:5]

In [ ]:
# step by step
# 1. load data
# 2. tokenize them
# 3. map ids (build vocabulary)
# 4. be happy

In [ ]:
tokens['en'] = [
    
]

In [ ]:
from collections import Counter

all_en_tokens = []
for t_list in tokens['en']:
    all_en_tokens.extend(t_list)

counter_en = Counter(all_en_tokens)
display(counter_en.most_common(10))

all_de_tokens = []
for t_list in tokens['de']:
    all_de_tokens.extend(t_list)

counter_de = Counter(all_de_tokens)
display(counter_de.most_common(10))

In [ ]:
len(set(all_en_tokens)), len(set(all_de_tokens))

In [13]:
vocab_en = [token for token in counter_en.keys()]
vocab_de = [token for token in counter_de.keys()]

NameError: name 'counter_en' is not defined

In [ ]:
vocab_en[:10], vocab_de[:10]

In [ ]:
vocab_en = ['<sos>', '<eos>'] + vocab_en
vocab_de = ['<sos>', '<eos>'] + vocab_de

In [ ]:
id2token_en = vocab_en
id2token_de = vocab_de

In [ ]:
id2token_en[10]

In [ ]:
token2id_en = {token: idx for idx, token in enumerate(id2token_en)}
token2id_de = {token: idx for idx, token in enumerate(id2token_de)}

## Dataset, Modelo

In [ ]:
import torch
from torch.utils.data import Dataset


class TranslationDataset(Dataset):
    def __init__(self, dataset_tokens, vocab_en, vocab_de):
        self.data_en = []
        self.data_de = []
        self.dataset_tokens = dataset_tokens

        for t_list in dataset_tokens['en']:
            indexes = [token2id_en[t] for t in t_list if t in vocab_en]
            self.data_en.append(indexes)

        for t_list in dataset_tokens['de']:
            t_list = ['<sos>'] + t_list + ['<eos>']
            indexes = [token2id_de[t] for t in t_list if t in vocab_de]
            self.data_de.append(indexes)

    def __len__(self):
        return len(self.dataset_tokens['en'])

    def __getitem__(self, idx):
        return torch.LongTensor(self.data_en[idx]), torch.LongTensor(self.data_de[idx])

In [ ]:
train_dataset = TranslationDataset(tokens, vocab_en, vocab_de)
test_dataset = TranslationDataset(test_tokens, vocab_en, vocab_de)

In [ ]:
len(train_dataset), len(test_dataset) # 200k, 70k

In [ ]:
from torch import nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers):
        super().__init__()

        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)

        return hidden, cell

In [ ]:
device = torch.device('cuda')

In [ ]:
INPUT_DIM = len(vocab_en)
EMB_DIM = 128
HID_DIM = 64
N_LAYERS = 1

enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS).to(device)

In [ ]:
x = train_dataset[0][0].to(device)
x

In [ ]:
output = enc(x)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers)

        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        embedded = self.embedding(input)
        outputs, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(outputs)

        return prediction, hidden, cell

## Entrada - Saída

tamanho do vocab -> 24000 tokens (unicos)  
teremos algo como  
[0.2, ..., 0.71, 0.0001, ...]  

escolhemos a posição com o maior valor


In [ ]:
INPUT_DIM = len(vocab_en)
EMB_DIM = 128
HID_DIM = 64
N_LAYERS = 1

dec = Decoder(INPUT_DIM, EMB_DIM, HID_DIM, N_LAYERS).to(device)

In [ ]:
y = train_dataset[0][1].to(device)
y

In [ ]:
dec_output = dec(torch.tensor(y), 0, 0)# nao funciona, ninguem liga

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, target, teacher_forcing_ratio=0.5):
        # não é normal passar a entrada + saída, mas vamos ajudar no treino para esse caso
        # so avisando
        source = source.to(self.device)
        target = target.to(self.device)

        target_len = target.shape[0]
        target_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(target_len, 1, target_vocab_size).to(self.device) # 1 magico, tem a ver com batches.
                                                                                # lembrando que nao estamos dividindo em batches
                                                                                # caso tivessemos batches, seria target_len, batch_size, target_vocab_size
        outputs[0][0][0] = 1 # O token inicial é o que tem índice 0.
                             # é 1 no id 0 e 0 no resto. Garantimos que é <sos>

        hidden, cell = self.encoder(source)

        input = target[0] # <sos>
        input = input.unsqueeze(0)

        for t in range(1, target_len):
            # teacher forcing ajuda muito no começo do treinamento
            # podemos usar estrategias de diminuir o ratio a medida que a epoca passa
            prediction, hidden, cell = self.decoder(input, hidden, cell)

            outputs[t] = prediction
            top1 = prediction.argmax(1)

            teacher_force = random.random() < teacher_forcing_ratio
            input = target[t].unsqueeze(0) if teacher_force else top1
        # ou geramos até ele chegar em um <eos>
        # ou geramos ate ele chegar no  numero da saída esperada
        # isso é importante porque no começo o decoder é bem burro, deixar ele gerar até <eos> pode dar
        #   em geração infinita.

        return outputs



In [ ]:
seq2seq = Seq2Seq(enc, dec, device).to(device)

In [ ]:
source, target = train_dataset[0]
source, target

In [ ]:
out = seq2seq(source, target, teacher_forcing_ratio=0.5)
out

In [ ]:
out.shape

In [ ]:
out.argmax(2)

In [ ]:
out[1][0][16688]

In [ ]:
id2token_de[0], id2token_de[16688], id2token_de[9324], id2token_de[1]

## Proximos passos

- fazer o loop de treino
- loop de teste